In [1]:
import numpy as np
import pandas as pd
import sys
PATH_TO_PREDICTOR_HOME = "../.."
sys.path.append(PATH_TO_PREDICTOR_HOME)
METHOD = "hmm_pomegranate"


# 1. Read data

In [2]:
DATA_PATH = 'D:/Google Drive/Work/ITMO/data/length_dependent_hmm/per_length_per_kfold_split/' # windows
# DATA_PATH = '/home/denklewer/Data/length_dependent_hmm/' #ubuntu
TARGET_PATH_TO_RESULTS = "experiment_results/iterative_alleles_2perposition_pc10/"

In [131]:
from data_reading_methods import get_available_alleles, read_data

ALLELES = list(sorted(get_available_alleles(DATA_PATH)))

per_allele_per_kfold_per_length_binders_train = read_data(DATA_PATH, ALLELES, "train", "binders")
per_allele_per_kfold_per_length_binders_test = read_data(DATA_PATH, ALLELES, "test", "binders")
per_allele_per_kfold_per_length_non_binders_train = read_data(DATA_PATH, ALLELES, "train", "nonbinders")
per_allele_per_kfold_per_length_non_binders_test = read_data(DATA_PATH, ALLELES, "test", "nonbinders")

ALLELES

['HLA-A*01:01', 'HLA-A*02:01', 'HLA-A*03:01', 'HLA-B*07:02', 'HLA-B*27:05']

# 2. Train HMMS

In [132]:
from hmm_logic_methods import train_model, save_model, add_more_states_and_reset_transitions


NUM_RUNS = 2
TARGET_LENGTHS= [9]
TOTAL_SPLITS = 5

In [133]:
start_length = 1

## 2.0 Merge alleles together

In [134]:
# combination_size = len(ALLELES)
ALLELES = ['HLA-A*01:01', 'HLA-A*03:01', 'HLA-B*07:02']
combination_size = 3

In [135]:
#Calculate median length
median_length = 5000
for allele_name in ALLELES:
    for split in range(TOTAL_SPLITS):
        for target_length in per_allele_per_kfold_per_length_binders_train[allele_name][split].keys():
            data_array = per_allele_per_kfold_per_length_binders_train[allele_name][split][target_length]
            print(f"length {target_length} was {len(data_array)}")
            per_allele_per_kfold_per_length_binders_train[allele_name][split][target_length] = data_array[:median_length]

length 8 was 764
length 9 was 5052
length 10 was 2464
length 11 was 1430
length 12 was 1106
length 13 was 842
length 14 was 568
length 15 was 464
length 16 was 345
length 17 was 209
length 18 was 128
length 19 was 75
length 20 was 54
length 21 was 29
length 22 was 21
length 23 was 17
length 24 was 16
length 25 was 8
length 26 was 7
length 28 was 6
length 29 was 4
length 30 was 5
length 8 was 764
length 9 was 5053
length 10 was 2465
length 11 was 1430
length 12 was 1106
length 13 was 842
length 14 was 568
length 15 was 465
length 16 was 345
length 17 was 209
length 18 was 129
length 19 was 75
length 20 was 54
length 21 was 29
length 22 was 21
length 23 was 17
length 24 was 16
length 25 was 8
length 26 was 7
length 28 was 6
length 29 was 5
length 30 was 5
length 8 was 764
length 9 was 5053
length 10 was 2465
length 11 was 1430
length 12 was 1106
length 13 was 842
length 14 was 568
length 15 was 465
length 16 was 346
length 17 was 210
length 18 was 129
length 19 was 75
length 20 was 54
le

In [136]:
import itertools
per_allele_per_kfold_per_length_binders_train_new = dict()
per_allele_per_kfold_per_length_binders_test_new = dict()
NEW_ALELLES = list()
selected_lengths = np.arange(8,20)
for combination in itertools.combinations(ALLELES, combination_size):
    allele_name = "+".join(combination)
    NEW_ALELLES.append(allele_name)
    print(allele_name)
    per_allele_per_kfold_per_length_binders_train_new[allele_name] = dict()
    per_allele_per_kfold_per_length_binders_test_new[allele_name] = dict()

    for split_num in range(TOTAL_SPLITS):
        per_allele_per_kfold_per_length_binders_train_new[allele_name][split_num] = dict()
        per_allele_per_kfold_per_length_binders_test_new[allele_name][split_num] = dict()

        list_of_length_sets = list()
        for old_allele in combination:
            lengths = set(per_allele_per_kfold_per_length_binders_train[old_allele][split_num].keys())
            list_of_length_sets.append(lengths)
        intersected_lengths = set.intersection(*list_of_length_sets)

        for target_length in intersected_lengths:
            data_arrays_train = list()
            data_arrays_test = list()
            if split_num == 0:
                print(f"Length {target_length}:" , end=" ")
            for old_allele in combination:
                single_allele_data_array_train = per_allele_per_kfold_per_length_binders_train[old_allele][split_num][target_length]
                if split_num == 0:
                    print(f"{old_allele} - {len(single_allele_data_array_train)}", end= " ")
                data_arrays_train.append(single_allele_data_array_train)

                single_allele_data_array_test = per_allele_per_kfold_per_length_binders_test[old_allele][split_num][target_length]
                if split_num == 0:
                    # print(f"{old_allele} - {len(single_allele_data_array_test)}", end= " ")
                    pass
                data_arrays_test.append(single_allele_data_array_test)
            print()

            per_allele_per_kfold_per_length_binders_train_new[allele_name][split_num][target_length] = np.concatenate(data_arrays_train)
            per_allele_per_kfold_per_length_binders_test_new[allele_name][split_num][target_length] = np.concatenate(data_arrays_test)




per_allele_per_kfold_per_length_binders_train = per_allele_per_kfold_per_length_binders_train_new
per_allele_per_kfold_per_length_binders_test = per_allele_per_kfold_per_length_binders_test_new
ALLELES = NEW_ALELLES

HLA-A*01:01+HLA-A*03:01+HLA-B*07:02
Length 8: HLA-A*01:01 - 764 HLA-A*03:01 - 93 HLA-B*07:02 - 1655 
Length 9: HLA-A*01:01 - 5000 HLA-A*03:01 - 5000 HLA-B*07:02 - 5000 
Length 10: HLA-A*01:01 - 2464 HLA-A*03:01 - 2184 HLA-B*07:02 - 3741 
Length 11: HLA-A*01:01 - 1430 HLA-A*03:01 - 691 HLA-B*07:02 - 1839 
Length 12: HLA-A*01:01 - 1106 HLA-A*03:01 - 202 HLA-B*07:02 - 803 
Length 13: HLA-A*01:01 - 842 HLA-A*03:01 - 98 HLA-B*07:02 - 530 
Length 14: HLA-A*01:01 - 568 HLA-A*03:01 - 31 HLA-B*07:02 - 381 
Length 15: HLA-A*01:01 - 464 HLA-A*03:01 - 18 HLA-B*07:02 - 338 
Length 16: HLA-A*01:01 - 345 HLA-A*03:01 - 12 HLA-B*07:02 - 188 
Length 17: HLA-A*01:01 - 209 HLA-A*03:01 - 5 HLA-B*07:02 - 137 
Length 18: HLA-A*01:01 - 128 HLA-A*03:01 - 4 HLA-B*07:02 - 76 














































## 2.1 Filter data

In [137]:
per_allele_per_kfold_per_length_binders_train_filtered = dict()
per_allele_per_kfold_per_length_binders_test_filtered = dict()


for allele_name in ALLELES:
    per_allele_per_kfold_per_length_binders_train_filtered[allele_name] = dict()
    per_allele_per_kfold_per_length_binders_test_filtered[allele_name] = dict()

    for split_num in per_allele_per_kfold_per_length_binders_train[allele_name].keys():
        per_allele_per_kfold_per_length_binders_train_filtered[allele_name][split_num] = dict()
        per_allele_per_kfold_per_length_binders_test_filtered[allele_name][split_num] = dict()

        for target_length in per_allele_per_kfold_per_length_binders_train[allele_name][split_num].keys():
            data_array = per_allele_per_kfold_per_length_binders_train[allele_name][split_num][target_length]
            per_allele_per_kfold_per_length_binders_train_filtered[allele_name][split_num][target_length] = np.array([s[:start_length] for s in data_array ])

            data_array = per_allele_per_kfold_per_length_binders_test[allele_name][split_num][target_length]
            per_allele_per_kfold_per_length_binders_test_filtered[allele_name][split_num][target_length] = np.array([s[:start_length] for s in data_array ])

## 2.2 Training process

In [138]:
stop_threshold = 0
cutoff = 0.0005
per_symbol_portion = -22 / 9
for target_length in TARGET_LENGTHS:
    target_length_for_training = start_length
    train_size = len(per_allele_per_kfold_per_length_binders_train[ALLELES[0]][0][target_length])
    ## Stop threshold Hardcoded here
    stop_threshold +=  abs(cutoff * train_size * per_symbol_portion  * target_length_for_training)
stop_threshold = stop_threshold / len(TARGET_LENGTHS)
stop_threshold

18.333333333333336

In [139]:
%%time
algorithm = 'viterbi'

per_name_models = dict()
per_allele_per_run_per_split_models=dict()
for allele_name in ALLELES:
    print(f"allele {allele_name}")
    per_allele_per_run_per_split_models[allele_name] = dict()
    for run_num in range(NUM_RUNS):
        print(f"run {run_num}")
        per_kfold_per_length_data =  per_allele_per_kfold_per_length_binders_train_filtered[allele_name]
        models, histories = train_model(allele_name,
                                        per_kfold_per_length_data,
                                        TARGET_LENGTHS,
                                        per_kfold_per_length_test_data=per_allele_per_kfold_per_length_binders_test_filtered[allele_name],
                                        verbose=True,
                                        multiple_check_input=False,
                                        algorithm=algorithm,
                                        name_prefix=f"{run_num}_run",
                                        num_states=4,
                                        n_jobs=10,
                                        stop_threshold=stop_threshold,
                                        min_iterations=5,
                                        emission_pseudocount=10
                                        )
        per_allele_per_run_per_split_models[allele_name][run_num] = models
        for split_num, model in models.items():
            per_name_models[model.name] = model
        # save models
        for split_num, model in models.items():
            history = histories[split_num]
            save_model(TARGET_PATH_TO_RESULTS, model,history)


allele HLA-A*01:01+HLA-A*03:01+HLA-B*07:02
run 0
split 0
train size 15000
Shuffling for fairness 
Simple training: Train Length 15000, init_batch_size None
[1] Improvement: 24475.43197917851	Time (s): 1.553
[2] Improvement: 0.0	Time (s): 1.413
[3] Improvement: 0.0	Time (s): 1.428
[4] Improvement: 0.0	Time (s): 1.736
[5] Improvement: 0.0	Time (s): 1.751
Total Training Improvement: 24475.43197917851
Total Training Time (s): 9.3600
run 1
split 0
train size 15000
Shuffling for fairness 
Simple training: Train Length 15000, init_batch_size None
[1] Improvement: 23667.95951276213	Time (s): 1.509
[2] Improvement: 0.0	Time (s): 1.376
[3] Improvement: 0.0	Time (s): 1.368
[4] Improvement: 0.0	Time (s): 1.431
[5] Improvement: 0.0	Time (s): 1.591
Total Training Improvement: 23667.95951276213
Total Training Time (s): 8.9020
Wall time: 18.5 s


In [140]:
from hmm_logic_methods import add_more_states_and_reset_transitions, add_more_states_to_the_end

In [141]:
per_allele_per_run_per_split_prepared_models = dict()
per_allele_per_run_per_split_original_models = dict()
for allele_name in ALLELES:
    per_allele_per_run_per_split_prepared_models[allele_name] = dict()
    per_allele_per_run_per_split_original_models[allele_name] = dict()
    for run_num in range(NUM_RUNS):
        per_allele_per_run_per_split_prepared_models[allele_name][run_num] = dict()
        per_allele_per_run_per_split_original_models[allele_name][run_num] = dict()
        for split_num, model in per_allele_per_run_per_split_models[allele_name][run_num].items():
            prepared_model = model.copy()
            prepared_model.name = prepared_model.name + "_iterations"
            per_allele_per_run_per_split_prepared_models[allele_name][run_num][split_num] = prepared_model
            per_allele_per_run_per_split_original_models[allele_name][run_num][split_num] = prepared_model

In [142]:
initial_length_for_training = start_length
max_length = max(TARGET_LENGTHS)

In [143]:
ADD_STATES_NUM = 4
for iteration in range(initial_length_for_training + 1,max_length+1):
    print(f"To length {iteration}")
    source_models = per_allele_per_run_per_split_prepared_models

    # copy models and rename
    per_allele_per_run_per_split_prepared_models = dict()
    for allele_name in ALLELES:
        print(f"allele {allele_name}")
        per_allele_per_run_per_split_prepared_models[allele_name] = dict()
        for run_num in range(NUM_RUNS):
            print(f"run {run_num}")
            per_allele_per_run_per_split_prepared_models[allele_name][run_num] = dict()
            for split_num, model in source_models[allele_name][run_num].items():
                print(f"split {split_num}")
                prepared_model = model.copy()
                old_states_names = [state.name for state in prepared_model.states]
                new_states_names =  state_names = ["s{:03d}".format(i) for i in range(len(prepared_model.states),len(prepared_model.states) + ADD_STATES_NUM)]
                original_model = per_allele_per_run_per_split_original_models[allele_name][run_num][split_num]
                new_model = add_more_states_to_the_end(prepared_model,
                                                                  ADD_STATES_NUM,
                                                                  new_sates_names =new_states_names,
                                                                  new_name = original_model.name + f"_to_length_{iteration}",
                                                                  new_probs="fair")

                print("Freezing the distribution for old states")
                print(f"old state {old_states_names}")
                frozen = []
                for state in new_model.states:
                    if (state.name in old_states_names) and state is not new_model.start and state is not new_model.end:
                        state.distribution.freeze()
                        frozen.append(state.name)
                print(f"freezed {frozen}")

                per_allele_per_run_per_split_prepared_models[allele_name][run_num][split_num] = new_model



    # make new data + 1 length
    per_allele_per_kfold_per_length_binders_train_filtered = dict()
    per_allele_per_kfold_per_length_binders_test_filtered = dict()

    for allele_name in ALLELES:
        per_allele_per_kfold_per_length_binders_train_filtered[allele_name] = dict()
        per_allele_per_kfold_per_length_binders_test_filtered[allele_name] = dict()

        for split_num in per_allele_per_kfold_per_length_binders_train[allele_name].keys():
            per_allele_per_kfold_per_length_binders_train_filtered[allele_name][split_num] = dict()
            per_allele_per_kfold_per_length_binders_test_filtered[allele_name][split_num] = dict()

            for target_length in per_allele_per_kfold_per_length_binders_train[allele_name][split_num].keys():
                data_array = per_allele_per_kfold_per_length_binders_train[allele_name][split_num][target_length]
                per_allele_per_kfold_per_length_binders_train_filtered[allele_name][split_num][target_length] = np.array([s[:iteration] for s in data_array ])

                data_array = per_allele_per_kfold_per_length_binders_test[allele_name][split_num][target_length]
                per_allele_per_kfold_per_length_binders_test_filtered[allele_name][split_num][target_length] = np.array([s[:iteration] for s in data_array ])
    # #train using new data
    # stop criteria correction
    for target_length in TARGET_LENGTHS:
        target_length_for_training = iteration
        train_size = len(per_allele_per_kfold_per_length_binders_train[ALLELES[0]][0][target_length])
        stop_threshold +=  abs(cutoff * train_size * per_symbol_portion  * target_length_for_training)
    stop_threshold = stop_threshold / len(TARGET_LENGTHS)

    per_allele_per_run_per_split_new_histories = dict()
    for allele_name in ALLELES:
        per_allele_per_run_per_split_new_histories[allele_name] = dict()
        for run_num in range(NUM_RUNS):
            per_kfold_per_length_data = per_allele_per_kfold_per_length_binders_train_filtered[allele_name]
            models, histories = train_model(allele_name,
                                            per_kfold_per_length_data,
                                            TARGET_LENGTHS,
                                            multiple_check_input=False,
                                            algorithm=algorithm,
                                            verbose=True,
                                        per_kfold_per_length_test_data=per_allele_per_kfold_per_length_binders_test_filtered[allele_name],
                                            stop_threshold=stop_threshold,
                                            min_iterations= 5,
                                            prepared_models=per_allele_per_run_per_split_prepared_models[allele_name][run_num], emission_pseudocount=10)
            per_allele_per_run_per_split_prepared_models[allele_name][run_num] = models
            per_allele_per_run_per_split_new_histories[allele_name][run_num] = histories
            # per_allele_per_run_per_split_models[allele_name][run_num] = models
            for split_num, model in models.items():
                per_name_models[model.name] = model
            # save models
            for split_num, model in models.items():
                history = histories[split_num]
                save_model(TARGET_PATH_TO_RESULTS, model,history)


To length 2
allele HLA-A*01:01+HLA-A*03:01+HLA-B*07:02
run 0
split 0
Initial length 6
num states to add 4
After length 10
 N is  8
sum from start 2.0
Freezing the distribution for old states
old state ['s000', 's001', 's002', 's003', '0_run-9_model-start', '0_run-9_model-end']
freezed ['s000', 's001', 's002', 's003']
run 1
split 0
Initial length 6
num states to add 4
After length 10
 N is  8
sum from start 2.0
Freezing the distribution for old states
old state ['s000', 's001', 's002', 's003', '1_run-9_model-start', '1_run-9_model-end']
freezed ['s000', 's001', 's002', 's003']
split 0
train size 15000
Shuffling for fairness 
Simple training: Train Length 15000, init_batch_size None
[1] Improvement: -4677.959786030115	Time (s): 0.86
[2] Improvement: 0.0	Time (s): 0.714
[3] Improvement: 0.0	Time (s): 0.769
[4] Improvement: 0.0	Time (s): 0.717
[5] Improvement: 0.0	Time (s): 0.777
Total Training Improvement: -4677.959786030115
Total Training Time (s): 4.6680
split 0
train size 15000
Shuffli

In [144]:
len(per_name_models.keys())

18

In [145]:
from hmm_visualization_methods import *

In [146]:
%%time

for i, model in enumerate(per_name_models.values()):
    print(i)
    # with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph.png", 'w+') as f:
    #     model.plot(file=f, crop_zero=False)
    with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph_cropped.png", 'w+') as f:
        model.plot(file=f, crop_zero=True)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
Wall time: 2.9 s


In [147]:
%%time

for i, model in enumerate(per_name_models.values()):
    plot_distributions_for_states(model,TARGET_PATH_TO_RESULTS, horizontal=False)
    print(i)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
Wall time: 3min 59s


In [148]:
%%time
per_name_pyviz = dict()
for name, model in per_name_models.items():
    per_name_pyviz[name] = make_pyviz_graph(model, TARGET_PATH_TO_RESULTS, precision=3)

Converting graph for model (0_run-9-4-st-viterbi-alg-5-min_iters_model-HLA_A_01_01+HLA_A_03_01+HLA_B_07_02-0) (Delete zero edges)
Converting graph for model (1_run-9-4-st-viterbi-alg-5-min_iters_model-HLA_A_01_01+HLA_A_03_01+HLA_B_07_02-0) (Delete zero edges)
Converting graph for model (0_run-9-4-st-viterbi-alg-5-min_iters_model-HLA_A_01_01+HLA_A_03_01+HLA_B_07_02-0_iterations_to_length_2) (Delete zero edges)
Converting graph for model (1_run-9-4-st-viterbi-alg-5-min_iters_model-HLA_A_01_01+HLA_A_03_01+HLA_B_07_02-0_iterations_to_length_2) (Delete zero edges)
Converting graph for model (0_run-9-4-st-viterbi-alg-5-min_iters_model-HLA_A_01_01+HLA_A_03_01+HLA_B_07_02-0_iterations_to_length_3) (Delete zero edges)
Converting graph for model (1_run-9-4-st-viterbi-alg-5-min_iters_model-HLA_A_01_01+HLA_A_03_01+HLA_B_07_02-0_iterations_to_length_3) (Delete zero edges)
Converting graph for model (0_run-9-4-st-viterbi-alg-5-min_iters_model-HLA_A_01_01+HLA_A_03_01+HLA_B_07_02-0_iterations_to_lengt

## Aux viterbi

In [124]:
target_length_for_training = 9
total_length = 9
cutoff = 0.00005
train_size = len(per_allele_per_kfold_per_length_binders_train[ALLELES[0]][0][9])
## Stop threshold Hardcoded here
stop_threshold =  abs(cutoff * train_size * -22/ total_length  * target_length_for_training)
per_allele_per_run_per_split_new_histories = dict()

for allele_name in ALLELES:
    for run_num in range(NUM_RUNS):
        for split_num, model in per_allele_per_run_per_split_prepared_models[allele_name][run_num].items():
            prepared_model = model.copy()
            prepared_model.name = prepared_model.name + "_plus_viterbi"

            for state in prepared_model.states:
                if state is not prepared_model.start and state is not prepared_model.end:
                    state.distribution.thaw()
            per_allele_per_run_per_split_prepared_models[allele_name][run_num][split_num] = prepared_model

In [125]:
per_name_new_models = dict()
for allele_name in ALLELES:
    per_allele_per_run_per_split_new_histories[allele_name] = dict()
    for run_num in range(NUM_RUNS):
        per_kfold_per_length_data = per_allele_per_kfold_per_length_binders_train_filtered[allele_name]
        models, histories = train_model(allele_name,
                                        per_kfold_per_length_data,
                                        TARGET_LENGTHS,
                                        multiple_check_input=False,
                                        algorithm='viterbi',
                                        verbose=True,
                                        per_kfold_per_length_test_data=per_allele_per_kfold_per_length_binders_test_filtered[allele_name],
                                        stop_threshold=stop_threshold,
                                        min_iterations= 5,
                                        prepared_models=per_allele_per_run_per_split_prepared_models[allele_name][run_num], emission_pseudocount=10)
        per_allele_per_run_per_split_prepared_models[allele_name][run_num] = models
        per_allele_per_run_per_split_new_histories[allele_name][run_num] = histories
        # per_allele_per_run_per_split_models[allele_name][run_num] = models
        for split_num, model in models.items():
            per_name_new_models[model.name] = model
        # save models
        for split_num, model in models.items():
            history = histories[split_num]
            save_model(TARGET_PATH_TO_RESULTS, model,history)

split 0
train size 15000
Shuffling for fairness 
Simple training: Train Length 15000, init_batch_size None
[1] Improvement: -2319.6268804584397	Time (s): 1.557
[2] Improvement: 814.229785465519	Time (s): 1.573
[3] Improvement: 401.4685630754684	Time (s): 1.538
[4] Improvement: 263.7783236054238	Time (s): 1.722
[5] Improvement: 129.05091817455832	Time (s): 3.115
[6] Improvement: 46.25211356021464	Time (s): 1.536
[7] Improvement: 68.05818232434103	Time (s): 1.67
[8] Improvement: 123.85719706863165	Time (s): 1.749
[9] Improvement: 31.023048596223816	Time (s): 1.588
[10] Improvement: -14.034395035589114	Time (s): 1.525
Total Training Improvement: -455.9431436236482
Total Training Time (s): 19.4120
split 0
train size 15000
Shuffling for fairness 
Simple training: Train Length 15000, init_batch_size None
[1] Improvement: -1803.4919827813283	Time (s): 1.704
[2] Improvement: 826.978028035257	Time (s): 1.658
[3] Improvement: 348.6208662634017	Time (s): 1.515
[4] Improvement: 286.8007785654627	T

In [126]:
%%time

for i, model in enumerate(per_name_new_models.values()):
    print(i)
    # with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph.png", 'w+') as f:
    #     model.plot(file=f, crop_zero=False)
    with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph_cropped.png", 'w+') as f:
        model.plot(file=f, crop_zero=True)

0
1
Wall time: 445 ms


In [127]:
%%time

for i, model in enumerate(per_name_new_models.values()):
    plot_distributions_for_states(model,TARGET_PATH_TO_RESULTS, horizontal=False)
    print(i)

0
1
Wall time: 47.7 s


In [128]:
per_name_pyviz = dict()
for name, model in per_name_new_models.items():
    per_name_pyviz[name] = make_pyviz_graph(model, TARGET_PATH_TO_RESULTS, precision=3)

Converting graph for model (0_run-9-4-st-baum-welch-alg-5-min_iters_model-HLA_A_01_01+HLA_A_03_01+HLA_B_07_02-0_iterations_to_length_9_plus_viterbi) (Delete zero edges)
new starting point
Converting graph for model (1_run-9-4-st-baum-welch-alg-5-min_iters_model-HLA_A_01_01+HLA_A_03_01+HLA_B_07_02-0_iterations_to_length_9_plus_viterbi) (Delete zero edges)
new starting point
new starting point
new starting point


In [54]:
model

{
    "class" : "HiddenMarkovModel",
    "name" : "1_run-9-5-st-baum-welch-alg-5-min_iters_model-HLA_A_01_01+HLA_A_02_01+HLA_A_03_01+HLA_B_07_02+HLA_B_27_05-0_iterations_to_length_9_plus_viterbi",
    "start" : {
        "class" : "State",
        "distribution" : null,
        "name" : "1_run-9-5-st-baum-welch-alg-5-min_iters_model-HLA_A_01_01+HLA_A_02_01+HLA_A_03_01+HLA_B_07_02+HLA_B_27_05-0_iterations_to_length_9-start",
        "weight" : 1.0
    },
    "end" : {
        "class" : "State",
        "distribution" : null,
        "name" : "1_run-9-5-st-baum-welch-alg-5-min_iters_model-HLA_A_01_01+HLA_A_02_01+HLA_A_03_01+HLA_B_07_02+HLA_B_27_05-0_iterations_to_length_9-end",
        "weight" : 1.0
    },
    "states" : [
        {
            "class" : "State",
            "distribution" : {
                "class" : "Distribution",
                "dtype" : "str",
                "name" : "DiscreteDistribution",
                "parameters" : [
                    {
                 

In [55]:
model.copy().dense_transition_matrix()

array([[0.00000000e+00, 6.11699855e-07, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       ...,
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 1.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 1.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00]])

In [24]:
model.states[37]

{
    "class" : "State",
    "distribution" : {
        "class" : "Distribution",
        "dtype" : "str",
        "name" : "DiscreteDistribution",
        "parameters" : [
            {
                "H" : 0.007322012774575479,
                "G" : 0.0,
                "I" : 0.0,
                "A" : 0.22152983330736875,
                "Q" : 0.0,
                "Y" : 0.05172145193955445,
                "L" : 0.15485278080697928,
                "S" : 0.0,
                "P" : 0.0,
                "E" : 0.0,
                "R" : 0.0,
                "M" : 0.0,
                "T" : 0.0,
                "W" : 0.0022589188347094565,
                "F" : 0.043231032871163734,
                "V" : 0.5190839694656488,
                "K" : 0.0,
                "D" : 0.0,
                "C" : 0.0,
                "N" : 0.0
            }
        ],
        "frozen" : false
    },
    "name" : "s039",
    "weight" : 1.0
}